# 07 — Evaluación del Pipeline

Comparamos los tres estados del dataset:
- **Original**: sin anomalías (ground truth)
- **Inyectado**: con anomalías sintéticas
- **Corregido**: después de aplicar los modelos

Las métricas RMSE/MAE son válidas aquí porque tenemos ground truth.

**Entrada:** parquets 01, 02, 06

In [ ]:
# ─── Intel Extension for Scikit-learn ────────────────────────────────────────
# IMPORTANTE: debe cargarse ANTES que cualquier import de sklearn.
# Instalar con: pip install scikit-learn-intelex
# En CPU Intel puede acelerar Random Forest e IterativeImputer 10-100x.
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    _intel_activo = True
except ImportError:
    _intel_activo = False

# ─── Librerías estándar ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os, glob, joblib, pickle, warnings
import matplotlib.ticker as mticker

# ─── Scikit-learn (ya parcheado si Intel Extension está disponible) ───────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.experimental import enable_iterative_imputer  # requerido antes de IterativeImputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix,
                              average_precision_score)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# ─── Configuración del proyecto ───────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from config import *

if _intel_activo:
    print("Intel Extension for Scikit-learn ACTIVA — algoritmos acelerados con oneAPI")
else:
    print("Tip: instala scikit-learn-intelex para acelerar en CPU Intel")
    print("     pip install scikit-learn-intelex")


## 0. Cargar los tres DataFrames

In [ ]:
df_original  = pd.read_parquet(PARQUET_01)
df_inyectado = pd.read_parquet(PARQUET_02)
df_corregido = pd.read_parquet(PARQUET_06)

print(f"Original : {df_original.shape}")
print(f"Inyectado: {df_inyectado.shape}")
print(f"Corregido: {df_corregido.shape}")

# ─── Aliases requeridos por las celdas de visualización y análisis ────────────
df_trabajo = df_inyectado                        # datos con anomalías inyectadas
df_contextual_corregido_custom = df_corregido    # datos corregidos

# datasets_listos: dict usado en celdas 11 y 13
nombre_objetivo = 'combined_2024_03_06-2025_11_30_1min'
datasets_listos = {nombre_objetivo: df_corregido}


## 1. Preparar índices temporales para comparación fila a fila

In [ ]:
print("--- Iniciando Preparación de Índices de DataFrames para Visualización ---")

dataframe_names = ["df_original", "df_trabajo", "df_contextual_corregido_custom"]

for df_name in dataframe_names:
    if df_name not in globals() or globals()[df_name] is None:
        print(f"DataFrame '{df_name}' no encontrado en memoria. Se omitirá su preparación.")
        continue

    # Obtener la referencia al DataFrame global para modificarlo directamente si es necesario
    df_instance = globals()[df_name] 
    original_shape = df_instance.shape # Guardar shape original para referencia
    print(f"Procesando DataFrame '{df_name}' (Shape inicial: {original_shape})...")
    
    try:
        # 1. Asegurar que 'Fecha' sea el índice y de tipo DatetimeIndex
        if isinstance(df_instance.index, pd.DatetimeIndex) and df_instance.index.name == 'Fecha':
            print(f"  '{df_name}' ya tiene 'Fecha' como DatetimeIndex.")
        elif 'Fecha' in df_instance.columns:
            print(f"  Configurando 'Fecha' como DatetimeIndex para '{df_name}'.")
            
            # Trabajar con una copia para manipulaciones de índice complejas si es necesario, pero el objetivo es actualizar el DataFrame global.
            current_df_object = df_instance.copy() # Usar copia para evitar SettingWithCopy en la variable iterada directamente
            
            # Si el índice se llama 'Fecha' pero no es Datetime, resetearlo antes de usar la columna 'Fecha'.
            if current_df_object.index.name == 'Fecha' and not isinstance(current_df_object.index, pd.DatetimeIndex):
                print(f"    Advertencia: '{df_name}' tiene un índice llamado 'Fecha' pero no es Datetime. Se reseteará el índice.")
                current_df_object = current_df_object.reset_index(drop=False) 
            
            # Asegurarse de que la columna 'Fecha' (si existe y se va a usar) sea datetime
            if 'Fecha' in current_df_object.columns:
                current_df_object['Fecha'] = pd.to_datetime(current_df_object['Fecha'])
                current_df_object.set_index('Fecha', inplace=True)
                globals()[df_name] = current_df_object # Actualizar la variable global explícitamente
                df_instance = current_df_object # Actualizar la referencia local para los siguientes pasos
            elif not isinstance(current_df_object.index, pd.DatetimeIndex) : # Si después de todo, no hay índice Fecha
                print(f"  ADVERTENCIA CRÍTICA: '{df_name}' no pudo ser configurado con DatetimeIndex 'Fecha' (columna 'Fecha' no encontrada o no se pudo usar).")
                continue # Saltar más procesamientos para este DF si el índice no es el esperado
            
        elif not isinstance(df_instance.index, pd.DatetimeIndex): # El DF no tiene columna 'Fecha' ni es DatetimeIndex
            print(f"  ADVERTENCIA: '{df_name}' no tiene columna 'Fecha' para convertir ni es DatetimeIndex. La visualización podría fallar.")
            continue

        # Utilizar la referencia global actualizada para los pasos de de-duplicación y ordenamiento
        df_to_process = globals()[df_name] 
        if isinstance(df_to_process.index, pd.DatetimeIndex): # Verificar de nuevo que sea DatetimeIndex
            if not df_to_process.index.is_unique:
                num_dupes_before = df_to_process.index.duplicated().sum()
                print(f"  ADVERTENCIA: El índice de '{df_name}' tiene {num_dupes_before} duplicados. Manteniendo la primera ocurrencia.")
                
                # SOLUCIÓN AL WARNING: Agregamos .copy() para desvincular el slice y evitar el SettingWithCopyWarning
                globals()[df_name] = df_to_process[~df_to_process.index.duplicated(keep='first')].copy()
                print(f"  '{df_name}' después de de-duplicar índice. Nuevo shape: {globals()[df_name].shape}")

            # Re-obtener la referencia por si cambió en el paso de de-duplicación
            df_to_sort = globals()[df_name] 
            if not df_to_sort.index.is_monotonic_increasing:
                print(f"  ADVERTENCIA: El índice de '{df_name}' no es monotónicamente creciente. Ordenando.")
                
                # SOLUCIÓN AL WARNING: Reasignamos el dataframe ordenado en lugar de usar inplace=True
                globals()[df_name] = df_to_sort.sort_index() 
    except Exception as e:
        print(f"  Error al procesar el índice para '{df_name}': {e}")

print("--- Preparación de Índices de DataFrames Completada ---")

# Re-obtener referencias finales a los DataFrames globales después de toda la preparación
df_original = globals().get('df_original')
df_trabajo = globals().get('df_trabajo')
df_contextual_corregido_custom = globals().get('df_contextual_corregido_custom')

# --- DEFINICIÓN SEGURA DE LAS COLUMNAS A VISUALIZAR ---
columnas_objetivo_existentes_global = globals().get('columnas_objetivo_existentes')
if not columnas_objetivo_existentes_global:
    print("ℹ️ 'columnas_objetivo_existentes' no encontrada en memoria. Usando lista por defecto.")
    columnas_objetivo_existentes_global = ['PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV', 'UVENT_cen', 'UVENT_lN', 'XCO2I', 'XHINV', 'XTINV', 'XTS']


# Verificar la preparación final de los DataFrames
df_original_ready = df_original is not None and isinstance(df_original.index, pd.DatetimeIndex)
df_trabajo_ready = df_trabajo is not None and isinstance(df_trabajo.index, pd.DatetimeIndex)
df_contextual_ready = df_contextual_corregido_custom is not None and isinstance(df_contextual_corregido_custom.index, pd.DatetimeIndex)

if not (df_original_ready and df_trabajo_ready and df_contextual_ready and \
        columnas_objetivo_existentes_global and \
        isinstance(columnas_objetivo_existentes_global, list) and \
        len(columnas_objetivo_existentes_global) > 0):
    print("\nError: DataFrames base ('df_original', 'df_trabajo', 'df_contextual_corregido_custom') o 'columnas_objetivo_existentes' no están disponibles, no tienen DatetimeIndex configurado correctamente, o la lista de columnas está vacía/no es válida. No se pueden generar las visualizaciones.")
else:
    import matplotlib.pyplot as plt
    for columna in columnas_objetivo_existentes_global:
        print(f"\n--- Visualizando columna: {columna} ---")
        
        original_col_exists = columna in df_original.columns
        trabajo_col_exists = columna in df_trabajo.columns
        contextual_col_exists = columna in df_contextual_corregido_custom.columns

        # --- Comparación 1: df_original vs df_trabajo ---
        if original_col_exists and trabajo_col_exists:
            # print(f"Comparación 1: df_original vs df_trabajo para {columna}") # Opcional: silenciado para menor ruido
            fig1, axes1 = plt.subplots(2, 1, figsize=(22, 10), sharex=True)
            fig1.suptitle(f"Comparación para: {columna}\n1. Original vs. Con Anomalías Inyectadas", fontsize=16)
            axes1[0].plot(df_original.index, df_original[columna], label="Original (df_original)", alpha=0.7, color="blue")
            axes1[0].set_title(f"{columna} - Datos Originales")
            axes1[0].set_ylabel(columna)
            axes1[0].legend(loc='upper left')
            axes1[0].grid(True, linestyle='--', alpha=0.5)
            axes1[1].plot(df_trabajo.index, df_trabajo[columna], label="Con Anomalías Inyectadas (df_trabajo)", linestyle="dashed", alpha=0.7, color="orange")
            if 'etiqueta_tipo_anomalia' in df_trabajo.columns:
                indices_con_anomalias_inyectadas = df_trabajo[df_trabajo['etiqueta_tipo_anomalia'].notna()].index
                common_indices_trabajo = df_trabajo.index.intersection(indices_con_anomalias_inyectadas)
                if not common_indices_trabajo.empty:
                    axes1[1].scatter(
                        df_trabajo.loc[common_indices_trabajo].index,
                        df_trabajo.loc[common_indices_trabajo, columna],
                        color='red', label='Anomalías Inyectadas', s=50, edgecolors='black', zorder=3 )
            axes1[1].set_title(f"{columna} - Datos con Anomalías Inyectadas")
            axes1[1].set_xlabel("Fecha")
            axes1[1].set_ylabel(columna)
            axes1[1].legend(loc='upper left')
            axes1[1].grid(True, linestyle='--', alpha=0.5)
            plt.tight_layout(rect=[0, 0.03, 1, 0.95]) 
            plt.show()
        else:
            print(f"  Skipping Comparación 1 para {columna}: Columna no encontrada.")

        # --- Comparación 2: df_original vs df_contextual_corregido_custom ---
        if original_col_exists and contextual_col_exists:
            # print(f"Comparación 2: df_original vs df_contextual_corregido_custom para {columna}")
            fig2, axes2 = plt.subplots(2, 1, figsize=(22, 10), sharex=True)
            fig2.suptitle(f"Comparación para: {columna}\n2. Original vs. Totalmente Corregido", fontsize=16)
            axes2[0].plot(df_original.index, df_original[columna], label="Original (df_original)", alpha=0.7, color="blue")
            axes2[0].set_title(f"{columna} - Datos Originales")
            axes2[0].set_ylabel(columna)
            axes2[0].legend(loc='upper left')
            axes2[0].grid(True, linestyle='--', alpha=0.5)
            axes2[1].plot(df_contextual_corregido_custom.index, df_contextual_corregido_custom[columna], label="Totalmente Corregido", alpha=0.7, color="green")
            axes2[1].set_title(f"{columna} - Datos Corregidos (Todas las etapas)")
            axes2[1].set_xlabel("Fecha")
            axes2[1].set_ylabel(columna)
            axes2[1].legend(loc='upper left')
            axes2[1].grid(True, linestyle='--', alpha=0.5)
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()
        else:
            print(f"  Skipping Comparación 2 para {columna}: Columna no encontrada.")

        # --- Comparación 3: df_trabajo vs df_contextual_corregido_custom ---
        if trabajo_col_exists and contextual_col_exists:
            # print(f"Comparación 3: df_trabajo vs df_contextual_corregido_custom para {columna}")
            fig3, axes3 = plt.subplots(2, 1, figsize=(22, 10), sharex=True)
            fig3.suptitle(f"Comparación para: {columna}\n3. Con Anomalías Inyectadas vs. Totalmente Corregido", fontsize=16)
            axes3[0].plot(df_trabajo.index, df_trabajo[columna], label="Con Anomalías Inyectadas (df_trabajo)", linestyle="dashed", alpha=0.7, color="orange")
            if 'etiqueta_tipo_anomalia' in df_trabajo.columns:
                indices_con_anomalias_inyectadas_trabajo = df_trabajo[df_trabajo['etiqueta_tipo_anomalia'].notna()].index
                common_indices_trabajo_comp3 = df_trabajo.index.intersection(indices_con_anomalias_inyectadas_trabajo)
                if not common_indices_trabajo_comp3.empty:
                    axes3[0].scatter(
                        df_trabajo.loc[common_indices_trabajo_comp3].index,
                        df_trabajo.loc[common_indices_trabajo_comp3, columna],
                        color='red', label='Anomalías Inyectadas', s=50, edgecolors='black', zorder=3 )
            axes3[0].set_title(f"{columna} - Datos con Anomalías Inyectadas")
            axes3[0].set_ylabel(columna)
            axes3[0].legend(loc='upper left')
            axes3[0].grid(True, linestyle='--', alpha=0.5)
            axes3[1].plot(df_contextual_corregido_custom.index, df_contextual_corregido_custom[columna], label="Totalmente Corregido", alpha=0.7, color="green")
            axes3[1].set_title(f"{columna} - Datos Corregidos (Todas las etapas)")
            axes3[1].set_xlabel("Fecha")
            axes3[1].set_ylabel(columna)
            axes3[1].legend(loc='upper left')
            axes3[1].grid(True, linestyle='--', alpha=0.5)
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()
        else:
            print(f"  Skipping Comparación 3 para {columna}: Columna no encontrada.")

print("\n--- Fin de la Visualización ---")

## 2. Análisis numérico: RMSE y MAE por columna

In [ ]:
print("--- Iniciando Análisis Numérico de Diferencias ---")

# Re-obtener referencias finales a los DataFrames globales
df_original = globals().get('df_original')
df_trabajo = globals().get('df_trabajo')
df_contextual_corregido_custom = globals().get('df_contextual_corregido_custom')

# --- MODIFICACIÓN IMPORTANTE: Validar y crear variables si no existen ---
columnas_objetivo_existentes_global = globals().get('columnas_objetivo_existentes')
if not columnas_objetivo_existentes_global:
    print("ℹ️ 'columnas_objetivo_existentes' no encontrada en memoria. Usando lista por defecto.")
    columnas_objetivo_existentes_global = ['PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV', 'UVENT_cen', 'UVENT_lN', 'XCO2I', 'XHINV', 'XTINV', 'XTS']

std_devs_originales = globals().get('std_devs_originales')
if std_devs_originales is None and df_original is not None:
    print("ℹ️ 'std_devs_originales' no encontrada en memoria. Calculándola automáticamente desde df_original.")
    # Calculamos la desviación estándar al vuelo para poder hacer la evaluación
    std_devs_originales = df_original[columnas_objetivo_existentes_global].std().to_dict()
# ------------------------------------------------------------------------

# Verificar la disponibilidad y preparación de los DataFrames
df_original_ready = df_original is not None and isinstance(df_original.index, pd.DatetimeIndex) and df_original.index.is_unique and df_original.index.is_monotonic_increasing
df_trabajo_ready = df_trabajo is not None and isinstance(df_trabajo.index, pd.DatetimeIndex) and df_trabajo.index.is_unique and df_trabajo.index.is_monotonic_increasing
df_contextual_ready = df_contextual_corregido_custom is not None and isinstance(df_contextual_corregido_custom.index, pd.DatetimeIndex) and df_contextual_corregido_custom.index.is_unique and df_contextual_corregido_custom.index.is_monotonic_increasing

if not (df_original_ready and df_trabajo_ready and df_contextual_ready and \
        columnas_objetivo_existentes_global and isinstance(columnas_objetivo_existentes_global, list) and len(columnas_objetivo_existentes_global) > 0 and \
        std_devs_originales is not None):
    print("\nError: DataFrames base no están listos (índice no es Datetime, único y ordenado), 'columnas_objetivo_existentes' o 'std_devs_originales' no disponibles. No se puede realizar el análisis numérico.")
else:
    print("\nDataFrames listos para el análisis numérico.")
    
    resumen_diferencias = []

    for columna in columnas_objetivo_existentes_global:
        print(f"\n--- Analizando columna: {columna} ---")

        # Asegurarse de que la columna exista en todos los DFs relevantes para las comparaciones
        if not (columna in df_original.columns and columna in df_trabajo.columns and columna in df_contextual_corregido_custom.columns):
            print(f"  Advertencia: La columna '{columna}' no está presente en todos los DataFrames necesarios. Saltando esta columna.")
            continue

        # --- 1. Conteo de Diferencias ---
        
        # Alineación y conteo: df_original vs df_trabajo
        common_idx_ot = df_original.index.intersection(df_trabajo.index)
        s_orig_ot = df_original.loc[common_idx_ot, columna]
        s_trab_ot = df_trabajo.loc[common_idx_ot, columna]
        
        # Contar diferencias: (val1 != val2) OR (uno es NaN y el otro no)
        # Llenar NaNs con un placeholder único para comparación directa si pd.NA no funciona como se espera con !=
        diff_mask_ot = (s_orig_ot != s_trab_ot) | (s_orig_ot.isnull() != s_trab_ot.isnull())
        count_diff_ot = diff_mask_ot.sum()
        print(f"  Número de valores diferentes entre df_original y df_trabajo (anomalías inyectadas): {count_diff_ot} / {len(common_idx_ot)}")

        # Alineación y conteo: df_original vs df_contextual_corregido_custom
        common_idx_oc = df_original.index.intersection(df_contextual_corregido_custom.index)
        s_orig_oc = df_original.loc[common_idx_oc, columna]
        s_corr_oc = df_contextual_corregido_custom.loc[common_idx_oc, columna]
        
        diff_mask_oc = (s_orig_oc != s_corr_oc) | (s_orig_oc.isnull() != s_corr_oc.isnull())
        count_diff_oc = diff_mask_oc.sum()
        print(f"  Número de valores diferentes entre df_original y df_contextual_corregido_custom: {count_diff_oc} / {len(common_idx_oc)}")

        # --- 2. Métricas de Diferencia (df_original vs df_contextual_corregido_custom) ---
        
        # Preparar datos para métricas (eliminar NaNs en pares)
        df_compare_oc = pd.DataFrame({'original': s_orig_oc, 'corregido': s_corr_oc}).dropna()
        
        if df_compare_oc.empty:
            print(f"  No hay datos comparables (después de dropna) entre df_original y df_contextual_corregido_custom para la columna {columna}.")
            rmse_overall = np.nan
            mae_overall = np.nan
            num_points_overall = 0
        else:
            rmse_overall = np.sqrt(mean_squared_error(df_compare_oc['original'], df_compare_oc['corregido']))
            mae_overall = mean_absolute_error(df_compare_oc['original'], df_compare_oc['corregido'])
            num_points_overall = len(df_compare_oc)
            print(f"  Métricas Generales (Original vs Corregido Total) para {num_points_overall} puntos:")
            print(f"    RMSE General: {rmse_overall:.4f}")
            print(f"    MAE General: {mae_overall:.4f}")

        # Métricas para puntos "muy diferentes"
        # Usar std_devs_originales[columna]
        std_col = std_devs_originales.get(columna)
        if std_col is None or std_col == 0: # Si no hay std_dev o es 0, no se puede usar para umbral
            print(f"  Advertencia: Desviación estándar para '{columna}' no disponible o es cero. No se calcularán métricas de diferencias significativas.")
            rmse_sig_2std = np.nan; mae_sig_2std = np.nan; count_sig_2std = 0
            rmse_sig_3std = np.nan; mae_sig_3std = np.nan; count_sig_3std = 0
        else:
            abs_diff = (df_compare_oc['original'] - df_compare_oc['corregido']).abs()
            
            # Umbral de 2 desviaciones estándar
            threshold_2std = 2 * std_col
            significant_diff_2std_mask = abs_diff > threshold_2std
            df_sig_diff_2std = df_compare_oc[significant_diff_2std_mask]
            count_sig_2std = len(df_sig_diff_2std)

            if count_sig_2std > 0:
                rmse_sig_2std = np.sqrt(mean_squared_error(df_sig_diff_2std['original'], df_sig_diff_2std['corregido']))
                mae_sig_2std = mean_absolute_error(df_sig_diff_2std['original'], df_sig_diff_2std['corregido'])
                print(f"  Métricas para {count_sig_2std} puntos 'muy diferentes' (> 2 std dev = {threshold_2std:.4f}):")
                print(f"    RMSE Significativo (2std): {rmse_sig_2std:.4f}")
                print(f"    MAE Significativo (2std): {mae_sig_2std:.4f}")
            else:
                print(f"  No se encontraron puntos con diferencias > 2 std dev para {columna}.")
                rmse_sig_2std = np.nan; mae_sig_2std = np.nan
            
            # Umbral de 3 desviaciones estándar
            threshold_3std = 3 * std_col
            significant_diff_3std_mask = abs_diff > threshold_3std
            df_sig_diff_3std = df_compare_oc[significant_diff_3std_mask]
            count_sig_3std = len(df_sig_diff_3std)

            if count_sig_3std > 0:
                rmse_sig_3std = np.sqrt(mean_squared_error(df_sig_diff_3std['original'], df_sig_diff_3std['corregido']))
                mae_sig_3std = mean_absolute_error(df_sig_diff_3std['original'], df_sig_diff_3std['corregido'])
                print(f"  Métricas para {count_sig_3std} puntos 'muy diferentes' (> 3 std dev = {threshold_3std:.4f}):")
                print(f"    RMSE Significativo (3std): {rmse_sig_3std:.4f}")
                print(f"    MAE Significativo (3std): {mae_sig_3std:.4f}")
            else:
                print(f"  No se encontraron puntos con diferencias > 3 std dev para {columna}.")
                rmse_sig_3std = np.nan; mae_sig_3std = np.nan

        resumen_diferencias.append({
            'Columna': columna,
            'Diff_Orig_vs_Trabajo': count_diff_ot,
            'Puntos_Comp_OT': len(common_idx_ot),
            'Diff_Orig_vs_CorregidoTotal': count_diff_oc,
            'Puntos_Comp_OC': len(common_idx_oc),
            'Puntos_Eval_Overall_OC': num_points_overall,
            'RMSE_Overall_OC': rmse_overall,
            'MAE_Overall_OC': mae_overall,
            'StdDev_Original_Col': std_col if std_col is not None else np.nan,
            'Puntos_SigDiff_2Std': count_sig_2std,
            'RMSE_SigDiff_2Std': rmse_sig_2std,
            'MAE_SigDiff_2Std': mae_sig_2std,
            'Puntos_SigDiff_3Std': count_sig_3std,
            'RMSE_SigDiff_3Std': rmse_sig_3std,
            'MAE_SigDiff_3Std': mae_sig_3std,
        })

    if resumen_diferencias:
        df_resumen_diferencias = pd.DataFrame(resumen_diferencias)
        print("\n\n--- Resumen del Análisis de Diferencias ---")
        # Configurar pandas para mostrar todas las columnas
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000) # Ajustar ancho para mejor visualización en consola
        print(df_resumen_diferencias.to_string())
        # Guardarlo como global si se necesita en el siguiente bloque
        globals()['df_resumen_diferencias'] = df_resumen_diferencias
    else:
        print("\nNo se generó resumen de diferencias (posiblemente no había columnas para analizar).")

print("\n--- Fin del Análisis Numérico de Diferencias ---")

## 3. Visualización de métricas

In [ ]:
# --- Inicio de la Visualización del Análisis Numérico ---

if 'df_resumen_diferencias' in globals() and isinstance(globals()['df_resumen_diferencias'], pd.DataFrame):
    df_resumen = globals()['df_resumen_diferencias'].copy()
    print("DataFrame 'df_resumen_diferencias' encontrado. Iniciando visualizaciones.")

    # --- 1. Número de Valores Diferentes por Sensor ---
    df_plot_diff_counts = df_resumen.set_index('Columna')[['Diff_Orig_vs_Trabajo', 'Diff_Orig_vs_CorregidoTotal']]
    # Ordenar para mejor visualización
    df_plot_diff_counts_sorted = df_plot_diff_counts.sort_values(by='Diff_Orig_vs_CorregidoTotal', ascending=False)
    
    ax1 = df_plot_diff_counts_sorted.plot(kind='bar', figsize=(15, 7), width=0.8)
    ax1.set_title('Número de Valores Diferentes por Sensor', fontsize=16)
    ax1.set_xlabel('Sensor', fontsize=12)
    ax1.set_ylabel('Cantidad de Diferencias', fontsize=12)
    ax1.legend(['Original vs. Trabajo (Anomalías Inyectadas)', 'Original vs. Corregido Total'])
    ax1.tick_params(axis='x', rotation=45) # Removido ha='right'
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: format(int(x), ','))) # Formatear eje Y
    plt.tight_layout()
    plt.show()

    # --- 2. Métricas Generales (RMSE y MAE) - Original vs. Corregido Total ---
    df_plot_overall_metrics = df_resumen.set_index('Columna')[['RMSE_Overall_OC', 'MAE_Overall_OC']]
    
    # RMSE General
    df_plot_rmse_overall_sorted = df_plot_overall_metrics[['RMSE_Overall_OC']].sort_values(by='RMSE_Overall_OC', ascending=False)
    ax2_rmse = df_plot_rmse_overall_sorted.plot(kind='bar', figsize=(15, 7), color='skyblue', legend=None)
    ax2_rmse.set_title('RMSE General (Original vs. Corregido Total)', fontsize=16)
    ax2_rmse.set_xlabel('Sensor', fontsize=12)
    ax2_rmse.set_ylabel('RMSE', fontsize=12)
    ax2_rmse.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

    # MAE General
    df_plot_mae_overall_sorted = df_plot_overall_metrics[['MAE_Overall_OC']].sort_values(by='MAE_Overall_OC', ascending=False)
    ax2_mae = df_plot_mae_overall_sorted.plot(kind='bar', figsize=(15, 7), color='lightcoral', legend=None)
    ax2_mae.set_title('MAE General (Original vs. Corregido Total)', fontsize=16)
    ax2_mae.set_xlabel('Sensor', fontsize=12)
    ax2_mae.set_ylabel('MAE', fontsize=12)
    ax2_mae.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

    # --- 3. Número de Puntos Significativamente Diferentes ---
    df_plot_sig_counts = df_resumen.set_index('Columna')[['Puntos_SigDiff_2Std', 'Puntos_SigDiff_3Std']]
    df_plot_sig_counts_sorted = df_plot_sig_counts.sort_values(by='Puntos_SigDiff_2Std', ascending=False)

    ax3 = df_plot_sig_counts_sorted.plot(kind='bar', figsize=(15, 7), width=0.8)
    ax3.set_title('Número de Puntos Significativamente Diferentes (Original vs. Corregido Total)', fontsize=16)
    ax3.set_xlabel('Sensor', fontsize=12)
    ax3.set_ylabel('Cantidad de Puntos', fontsize=12)
    ax3.legend(['Diferencia > 2 Std Dev', 'Diferencia > 3 Std Dev'])
    ax3.tick_params(axis='x', rotation=45)
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: format(int(x), ',')))
    plt.tight_layout()
    plt.show()

    # --- 4. RMSE para Puntos Significativamente Diferentes ---
    df_plot_rmse_sig = df_resumen.set_index('Columna')[['RMSE_SigDiff_2Std', 'RMSE_SigDiff_3Std']]
    df_plot_rmse_sig_sorted = df_plot_rmse_sig.sort_values(by='RMSE_SigDiff_2Std', ascending=False)
    
    ax4 = df_plot_rmse_sig_sorted.plot(kind='bar', figsize=(15, 7), width=0.8)
    ax4.set_title('RMSE para Puntos Significativamente Diferentes (Original vs. Corregido Total)', fontsize=16)
    ax4.set_xlabel('Sensor', fontsize=12)
    ax4.set_ylabel('RMSE', fontsize=12)
    ax4.legend(['RMSE para Puntos con Diff > 2 Std Dev', 'RMSE para Puntos con Diff > 3 Std Dev'])
    ax4.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

    # --- 5. MAE para Puntos Significativamente Diferentes ---
    df_plot_mae_sig = df_resumen.set_index('Columna')[['MAE_SigDiff_2Std', 'MAE_SigDiff_3Std']]
    df_plot_mae_sig_sorted = df_plot_mae_sig.sort_values(by='MAE_SigDiff_2Std', ascending=False)

    ax5 = df_plot_mae_sig_sorted.plot(kind='bar', figsize=(15, 7), width=0.8)
    ax5.set_title('MAE para Puntos Significativamente Diferentes (Original vs. Corregido Total)', fontsize=16)
    ax5.set_xlabel('Sensor', fontsize=12)
    ax5.set_ylabel('MAE', fontsize=12)
    ax5.legend(['MAE para Puntos con Diff > 2 Std Dev', 'MAE para Puntos con Diff > 3 Std Dev'])
    ax5.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

    print("\n--- Fin de la Visualización del Análisis Numérico ---")

else:
    print("Error: No se encontró el DataFrame 'df_resumen_diferencias' en el entorno global o no es un DataFrame. Por favor, asegúrate de que se haya generado correctamente en el paso anterior.")

## 4. Gráficas comparativas por sensor

In [ ]:
print("=== INICIANDO PREPARACIÓN DE VISUALIZACIONES ===")

nombre_objetivo = 'combined_2024_03_06-2025_11_30_1min'

# --- 1. CARGA SEGURA DE LOS 3 DATAFRAMES ---
# A. Original (Sin daños)
df_plot_orig = df_original.copy()
# B. Inyectado (El que evaluó el Modelo 2, antes de corregir. Se guardó como 'df_trabajo' en la Fase 1)
df_plot_inyectado = df_trabajo.copy() 
# C. Totalmente Corregido (El resultado final de nuestra Fase 6)
df_plot_corregido = datasets_listos[nombre_objetivo].copy()

# Función auxiliar para garantizar que el índice sea de Tiempo (Datetime) y esté limpio
def preparar_indice_tiempo(df, nombre):
    if 'Fecha' in df.columns and not isinstance(df.index, pd.DatetimeIndex):
        df['Fecha'] = pd.to_datetime(df['Fecha'], errors='coerce')
        df.set_index('Fecha', inplace=True)
    if isinstance(df.index, pd.DatetimeIndex):
        df = df[~df.index.duplicated(keep='first')].sort_index()
    else:
        print(f"⚠️ Advertencia: {nombre} no pudo convertirse a DatetimeIndex.")
    return df

df_plot_orig = preparar_indice_tiempo(df_plot_orig, "df_original")
df_plot_inyectado = preparar_indice_tiempo(df_plot_inyectado, "df_trabajo (Inyectado)")
df_plot_corregido = preparar_indice_tiempo(df_plot_corregido, "df_corregido")

# --- 2. DEFINICIÓN DE COLUMNAS A VISUALIZAR ---
columnas_objetivo = ['PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV', 
                     'XCO2I', 'XHINV', 'XTINV', 'XTS', 'UVENT_cen', 'UVENT_lN']

columnas_a_graficar = [col for col in columnas_objetivo if col in df_plot_orig.columns and col in df_plot_corregido.columns]

# --- 3. GENERACIÓN DE GRÁFICAS ---
for columna in columnas_a_graficar:
    print(f"\n--- Generando gráficas para: {columna} ---")
    
    # Preparamos los puntos rojos (Anomalías detectadas por el Modelo 2)
    puntos_anomalos_x = []
    puntos_anomalos_y = []
    if 'etiqueta_tipo_anomalia' in df_plot_inyectado.columns:
        # Filtramos solo las filas que NO son nulas en la etiqueta de anomalía
        mask_anomalias = df_plot_inyectado['etiqueta_tipo_anomalia'].notna()
        puntos_anomalos_x = df_plot_inyectado[mask_anomalias].index
        puntos_anomalos_y = df_plot_inyectado.loc[mask_anomalias, columna]

    # --- FIGURA 1: Original vs Inyectado ---
    fig1, axes1 = plt.subplots(2, 1, figsize=(20, 8), sharex=True)
    fig1.suptitle(f"1. IMPACTO DE ANOMALÍAS - Variable: {columna}", fontsize=16, fontweight='bold')
    
    axes1[0].plot(df_plot_orig.index, df_plot_orig[columna], label="Datos Originales (Sanos)", color="blue", alpha=0.8)
    axes1[0].set_title("Comportamiento Original")
    axes1[0].set_ylabel(columna)
    axes1[0].legend(loc='upper left')
    axes1[0].grid(True, linestyle='--', alpha=0.5)
    
    axes1[1].plot(df_plot_inyectado.index, df_plot_inyectado[columna], label="Datos Dañados", color="orange", alpha=0.8)
    if len(puntos_anomalos_x) > 0:
        axes1[1].scatter(puntos_anomalos_x, puntos_anomalos_y, color='red', label='Anomalía Detectada (Modelo 2)', s=40, edgecolor='black', zorder=3)
    axes1[1].set_title("Comportamiento con Anomalías Inyectadas")
    axes1[1].set_ylabel(columna)
    axes1[1].legend(loc='upper left')
    axes1[1].grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

    # --- FIGURA 2: Original vs Totalmente Corregido ---
    fig2, axes2 = plt.subplots(2, 1, figsize=(20, 8), sharex=True)
    fig2.suptitle(f"2. EFECTIVIDAD DE CORRECCIÓN - Variable: {columna}", fontsize=16, fontweight='bold')
    
    axes2[0].plot(df_plot_orig.index, df_plot_orig[columna], label="Datos Originales (Sanos)", color="blue", alpha=0.8)
    axes2[0].set_title("Comportamiento Original")
    axes2[0].set_ylabel(columna)
    axes2[0].legend(loc='upper left')
    axes2[0].grid(True, linestyle='--', alpha=0.5)
    
    axes2[1].plot(df_plot_corregido.index, df_plot_corregido[columna], label="Datos Corregidos (Modelo 3)", color="green", alpha=0.8)
    axes2[1].set_title("Comportamiento Reconstruido")
    axes2[1].set_ylabel(columna)
    axes2[1].legend(loc='upper left')
    axes2[1].grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

    # --- FIGURA 3: Inyectado vs Totalmente Corregido ---
    fig3, axes3 = plt.subplots(2, 1, figsize=(20, 8), sharex=True)
    fig3.suptitle(f"3. PROCESO DE LIMPIEZA - Variable: {columna}", fontsize=16, fontweight='bold')
    
    axes3[0].plot(df_plot_inyectado.index, df_plot_inyectado[columna], label="Datos Dañados", color="orange", alpha=0.8)
    if len(puntos_anomalos_x) > 0:
        axes3[0].scatter(puntos_anomalos_x, puntos_anomalos_y, color='red', label='Anomalía Detectada', s=40, edgecolor='black', zorder=3)
    axes3[0].set_title("Antes de la Corrección")
    axes3[0].set_ylabel(columna)
    axes3[0].legend(loc='upper left')
    axes3[0].grid(True, linestyle='--', alpha=0.5)
    
    axes3[1].plot(df_plot_corregido.index, df_plot_corregido[columna], label="Datos Corregidos (Modelo 3)", color="green", alpha=0.8)
    axes3[1].set_title("Después de la Corrección")
    axes3[1].set_xlabel("Fecha")
    axes3[1].set_ylabel(columna)
    axes3[1].legend(loc='upper left')
    axes3[1].grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

print("\n=== FIN DE LA VISUALIZACIÓN ===")

## 5. Tabla de diferencias numéricas

In [ ]:
print("=== INICIANDO ANÁLISIS NUMÉRICO DE DIFERENCIAS ===")

nombre_objetivo = 'combined_2024_03_06-2025_11_30_1min'

# --- 1. CARGA Y PREPARACIÓN CON REDONDEO (Sincronización Crítica) ---
df_orig = df_original.copy()
df_inyectado = df_trabajo.copy()
df_corregido = datasets_listos[nombre_objetivo].copy()

def preparar_y_sincronizar(df):
    if 'Fecha' in df.columns:
        df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')
        df = df.set_index('Fecha')
    # REDONDEO AL MINUTO: Vital para que la intersección no devuelva 0 filas
    df.index = pd.to_datetime(df.index, errors='coerce').round('1min')
    # Limpieza de duplicados y ordenamiento
    return df[~df.index.duplicated(keep='first')].sort_index()

df_orig = preparar_y_sincronizar(df_orig)
df_inyectado = preparar_y_sincronizar(df_inyectado)
df_corregido = preparar_y_sincronizar(df_corregido)

# Columnas a evaluar
columnas_validas = [col for col in columnas_objetivo if col in df_orig.columns and col in df_corregido.columns]

resumen_diferencias = []

# --- 2. ANÁLISIS COLUMNA POR COLUMNA ---
for columna in columnas_validas:
    # Intersección de los 3 índices para comparación triple exacta
    idx_comun = df_orig.index.intersection(df_inyectado.index).intersection(df_corregido.index)
    
    if idx_comun.empty:
        print(f"⚠️ Advertencia: No hay coincidencia de fechas para {columna}.")
        continue

    s_orig = df_orig.loc[idx_comun, columna]
    s_inyect = df_inyectado.loc[idx_comun, columna]
    s_corr = df_corregido.loc[idx_comun, columna]
    
    # A. Conteo de celdas modificadas (ignora si ambos son NaN)
    # Usamos np.isclose o una tolerancia pequeña si son floats, o != si son precisos
    mask_daño = (s_orig != s_inyect) & ~(s_orig.isna() & s_inyect.isna())
    mask_residuo = (s_orig != s_corr) & ~(s_orig.isna() & s_corr.isna())
    
    diff_inyectadas = mask_daño.sum()
    diff_restantes = mask_residuo.sum()
    
    # B. Métricas Globales (Solo sobre puntos con datos en ambos)
    mask_valid = s_orig.notna() & s_corr.notna()
    if mask_valid.any():
        s_o_v = s_orig[mask_valid]
        s_c_v = s_corr[mask_valid]
        
        std_col = s_o_v.std()
        rmse_overall = np.sqrt(mean_squared_error(s_o_v, s_c_v))
        mae_overall = mean_absolute_error(s_o_v, s_c_v)
        
        # C. Errores Significativos (> 2 y 3 Desviaciones Estándar)
        abs_diff = (s_o_v - s_c_v).abs()
        
        # Umbral 2 STD
        mask_2std = abs_diff > (2 * std_col)
        c_2std = mask_2std.sum()
        r_2std = np.sqrt(mean_squared_error(s_o_v[mask_2std], s_c_v[mask_2std])) if c_2std > 0 else 0
        
        # Umbral 3 STD
        mask_3std = abs_diff > (3 * std_col)
        c_3std = mask_3std.sum()
        r_3std = np.sqrt(mean_squared_error(s_o_v[mask_3std], s_c_v[mask_3std])) if c_3std > 0 else 0
        
        resumen_diferencias.append({
            'Columna': columna,
            'Daños_Inyectados': diff_inyectadas,
            'Diferencias_Restantes': diff_restantes,
            'Puntos_Eval': len(s_o_v),
            'RMSE_General': round(rmse_overall, 4),
            'MAE_General': round(mae_overall, 4),
            'Err_>2Std_(Cant)': c_2std,
            'RMSE_>2Std': round(r_2std, 4),
            'Err_>3Std_(Cant)': c_3std,
            'RMSE_>3Std': round(r_3std, 4)
        })

# --- 3. TABLA DE RESUMEN FINAL ---
if resumen_diferencias:
    df_resumen = pd.DataFrame(resumen_diferencias)
    print("\n" + "="*80)
    print("                 RESUMEN FINAL DE CALIDAD DE CORRECCIÓN")
    print("="*80)
    print(df_resumen.to_string(index=False))
    df_metricas_finales = df_resumen 
else:
    print("\n❌ Error: No se pudieron sincronizar los datos para el análisis.")

print("\n=== FIN DEL ANÁLISIS NUMÉRICO ===")

## 6. Visualización de la tabla

In [ ]:
print("=== INICIANDO VISUALIZACIÓN DEL ANÁLISIS NUMÉRICO ===")

# Verificamos que el DataFrame del paso anterior exista
if 'df_metricas_finales' in locals() or 'df_metricas_finales' in globals():
    df_resumen = df_metricas_finales.copy()
    print("-> Tabla de métricas encontrada. Generando gráficas...\n")

    # Configuración visual global
    plt.style.use('default')

    # --- 1. Progreso de Limpieza: Daños vs Restantes ---
    # Ajustamos a los nombres de columna del bloque anterior: 'Daños_Inyectados' y 'Diferencias_Restantes'
    df_plot_diff = df_resumen.set_index('Columna')[['Daños_Inyectados', 'Diferencias_Restantes']]
    df_plot_diff = df_plot_diff.sort_values(by='Diferencias_Restantes', ascending=False)
    
    ax1 = df_plot_diff.plot(kind='bar', figsize=(15, 6), width=0.8, color=['#FFA07A', '#20B2AA'])
    ax1.set_title('1. Progreso de Limpieza: Daños Inyectados vs Diferencias Restantes', fontsize=16, fontweight='bold')
    ax1.set_xlabel('Sensor', fontsize=12)
    ax1.set_ylabel('Cantidad de Celdas', fontsize=12)
    ax1.legend(['Daños Iniciales (Inyectados)', 'Errores Residuales (Post-Corrección)'])
    ax1.tick_params(axis='x', rotation=45)
    # Formateador de miles para legibilidad
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: format(int(x), ',')))
    ax1.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    # --- 2. Métricas Generales (RMSE y MAE) ---
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
    # RMSE General
    df_rmse = df_resumen.set_index('Columna')[['RMSE_General']].sort_values(by='RMSE_General', ascending=False)
    df_rmse.plot(kind='bar', ax=axes[0], color='skyblue', legend=False)
    axes[0].set_title('2A. Error Cuadrático Medio (RMSE) General', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('RMSE', fontsize=12)
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(axis='y', linestyle='--', alpha=0.7)
    
    # MAE General
    df_mae = df_resumen.set_index('Columna')[['MAE_General']].sort_values(by='MAE_General', ascending=False)
    df_mae.plot(kind='bar', ax=axes[1], color='lightcoral', legend=False)
    axes[1].set_title('2B. Error Absoluto Medio (MAE) General', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('MAE', fontsize=12)
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

    # --- 3. Cantidad de Errores Graves ---
    # Ajustado a: 'Err_>2Std_(Cant)' y 'Err_>3Std_(Cant)'
    df_plot_sig = df_resumen.set_index('Columna')[['Err_>2Std_(Cant)', 'Err_>3Std_(Cant)']]
    df_plot_sig = df_plot_sig.sort_values(by='Err_>2Std_(Cant)', ascending=False)

    ax3 = df_plot_sig.plot(kind='bar', figsize=(15, 6), width=0.8, color=['#FFD700', '#DC143C'])
    ax3.set_title('3. Frecuencia de Errores Graves (>2 y >3 Desviaciones Estándar)', fontsize=16, fontweight='bold')
    ax3.set_xlabel('Sensor', fontsize=12)
    ax3.set_ylabel('Cantidad de Puntos', fontsize=12)
    ax3.legend(['Error > 2 Std Dev', 'Error > 3 Std Dev'])
    ax3.tick_params(axis='x', rotation=45)
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: format(int(x), ',')))
    ax3.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    # --- 4. RMSE de Errores Graves ---
    # Ajustado a: 'RMSE_>2Std' y 'RMSE_>3Std'
    df_plot_rmse_sig = df_resumen.set_index('Columna')[['RMSE_>2Std', 'RMSE_>3Std']]
    df_plot_rmse_sig = df_plot_rmse_sig.sort_values(by='RMSE_>2Std', ascending=False)
    
    ax4 = df_plot_rmse_sig.plot(kind='bar', figsize=(15, 6), width=0.8, color=['#DAA520', '#8B0000'])
    ax4.set_title('4. Magnitud del Error (RMSE) en Puntos Críticos', fontsize=16, fontweight='bold')
    ax4.set_xlabel('Sensor', fontsize=12)
    ax4.set_ylabel('RMSE', fontsize=12)
    ax4.legend(['RMSE en Errores > 2 Std Dev', 'RMSE en Errores > 3 Std Dev'])
    ax4.tick_params(axis='x', rotation=45)
    ax4.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    print("\n=== FIN DE LA VISUALIZACIÓN DEL ANÁLISIS NUMÉRICO ===")

else:
    print("❌ Error: No se encontró la tabla 'df_metricas_finales'.")